In [1]:
from pyspark.sql import functions as F
from datetime import datetime

run_ts = datetime.now()
results = []

def log(check, table, bad_rows):
    results.append((run_ts, check, table, int(bad_rows), "PASS" if bad_rows == 0 else "FAIL"))

fact = spark.read.table("gold.fact_sales")
cust = spark.read.table("gold.dim_customers")
prod = spark.read.table("gold.dim_products")

# --- fact_sales checks ---
log("NULL customer_key", "gold.fact_sales",
    fact.filter(F.col("customer_key").isNull()).count())

log("NULL product_key", "gold.fact_sales",
    fact.filter(F.col("product_key").isNull()).count())

log("Fully duplicated rows", "gold.fact_sales",
    fact.count() - fact.dropDuplicates().count())

log("Sales amount <= 0", "gold.fact_sales",
    fact.filter(F.col("sales_amount") <= 0).count())

log("Sales != quantity x price", "gold.fact_sales",
    fact.filter(F.abs(F.col("sales_amount") - F.col("quantity") * F.col("price")) > 0.01).count())

log("Ship date before order date", "gold.fact_sales",
    fact.filter(F.col("ship_date") < F.col("order_date")).count())

# --- orphan keys (fact rows with no matching dimension row) ---
log("Orphan customer_key", "gold.fact_sales",
    fact.filter(F.col("customer_key").isNotNull())
        .join(cust, "customer_key", "left_anti").count())

log("Orphan product_key", "gold.fact_sales",
    fact.filter(F.col("product_key").isNotNull())
        .join(prod, "product_key", "left_anti").count())

# --- dimension checks ---
log("Duplicate customer_key", "gold.dim_customers",
    cust.count() - cust.select("customer_key").distinct().count())

log("Duplicate product_key", "gold.dim_products",
    prod.count() - prod.select("product_key").distinct().count())

# --- save results ---
spark.sql("CREATE SCHEMA IF NOT EXISTS dq")

schema = "run_time timestamp, check_name string, table_name string, bad_rows long, status string"
df = spark.createDataFrame(results, schema)
df.write.format("delta").mode("append").saveAsTable("dq.data_quality_log")

display(df)

StatementMeta(, 21f95968-8783-4f24-8ef4-30ae4d549c31, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dd0e51ac-d126-498b-acb8-88b912b0ace0)